# 🧠 Smart Customer Behavior Analysis
### RFM Analysis + K-Means Clustering
---
> **Goal:** Segment customers based on their purchase behavior to drive targeted marketing strategies.

## 📦 Step 1 — Import Libraries & Load Data

In [ ]:
# 📦 Core data libraries
import pandas as pd
import numpy as np
import datetime as dt

# 🎨 Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 🤖 Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ⚙️ Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

# 🎨 Set global plot style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor': '#2a2a3e',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'text.color': 'white',
    'axes.titlecolor': 'white',
    'grid.color': '#444466',
    'axes.edgecolor': '#555577'
})

print('✅ All libraries imported successfully!')

In [ ]:
# 📂 Load the raw transaction data
df = pd.read_csv('data/transactions.csv')

print(f'📊 Raw dataset shape: {df.shape}')
print(f'📋 Columns: {list(df.columns)}')
df.head()

## 🔍 Step 2 — Exploratory Data Analysis

In [ ]:
# 🔍 Quick data health check before cleaning
print('=' * 50)
print('📈 DATASET STATISTICS')
print('=' * 50)
print(df.describe().round(2))

print('\n' + '=' * 50)
print('🚨 MISSING VALUES')
print('=' * 50)
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else '✅ No missing values found!')

In [ ]:
# 📊 Visualize raw data distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📊 Raw Data Distribution Overview', fontsize=16, fontweight='bold', color='white', y=1.02)

colors = ['#7c3aed', '#0ea5e9', '#10b981']

# 💰 Amount distribution (log scale for heavy-tailed data)
axes[0].hist(df['Amount'].clip(lower=0), bins=60, color=colors[0], alpha=0.85, edgecolor='white', linewidth=0.3)
axes[0].set_title('💰 Transaction Amount', fontweight='bold')
axes[0].set_xlabel('Amount (£)')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

# 📦 Quantity distribution
axes[1].hist(df['Quantity'].clip(lower=0, upper=df['Quantity'].quantile(0.98)), bins=60,
             color=colors[1], alpha=0.85, edgecolor='white', linewidth=0.3)
axes[1].set_title('📦 Quantity per Transaction', fontweight='bold')
axes[1].set_xlabel('Quantity')
axes[1].set_ylabel('Count')

# 🌍 Top 10 countries
top_countries = df['Country'].value_counts().head(10)
axes[2].barh(top_countries.index[::-1], top_countries.values[::-1], color=colors[2], alpha=0.85)
axes[2].set_title('🌍 Top 10 Countries', fontweight='bold')
axes[2].set_xlabel('Transaction Count')

plt.tight_layout()
plt.show()

## 🧹 Step 3 — Data Cleaning

In [ ]:
# 🧹 Drop rows with any missing values
before = len(df)
df = df.dropna()
after = len(df)

print(f'🗑️  Removed {before - after:,} rows with missing values')
print(f'✅ Clean dataset: {after:,} rows remaining')

# 💾 Save cleaned data
df.to_csv('data/transactions_clean.csv', index=False)
print('💾 Cleaned CSV saved to data/transactions_clean.csv')

## 📐 Step 4 — RFM Feature Engineering

| Metric | Description |
|---|---|
| **R**ecency | How recently did the customer buy? (lower = better) |
| **F**requency | How often do they buy? (higher = better) |
| **M**onetary | How much do they spend? (higher = better) |

In [ ]:
# 📅 Parse dates — required for recency calculation
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'📅 Date range: {df["InvoiceDate"].min().date()}  →  {df["InvoiceDate"].max().date()}')

In [ ]:
# 📌 Snapshot date = 1 day after the last transaction
# (simulates "today" for recency calculation)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
print(f'📌 Snapshot date (reference point): {snapshot_date.date()}')

In [ ]:
# 🏗️ Compute RFM metrics per customer
rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),  # days since last purchase
    Frequency=('Invoice', 'count'),                                      # total number of transactions
    Monetary=('Amount', 'sum')                                           # total spend
).reset_index()

print(f'👥 Total unique customers: {len(rfm):,}')
print('\n📋 RFM Table (first 5 rows):')
rfm.head()

In [ ]:
# 📊 Visualize RFM distributions before cleaning
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📐 RFM Feature Distributions (Raw)', fontsize=16, fontweight='bold', color='white', y=1.02)

palette = ['#f472b6', '#38bdf8', '#4ade80']
labels = ['Recency (days)', 'Frequency (transactions)', 'Monetary (£)']
cols = ['Recency', 'Frequency', 'Monetary']

for ax, col, color, label in zip(axes, cols, palette, labels):
    data = rfm[col].clip(upper=rfm[col].quantile(0.99))
    ax.hist(data, bins=50, color=color, alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.axvline(rfm[col].median(), color='yellow', linestyle='--', linewidth=1.5, label=f'Median: {rfm[col].median():.0f}')
    ax.set_title(f'📊 {col}', fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 🚿 Remove extreme monetary outliers (top 1%) to improve clustering
q99 = rfm['Monetary'].quantile(0.99)
rfm_clean = rfm[rfm['Monetary'] < q99].copy()

removed = len(rfm) - len(rfm_clean)
print(f'🔪 Removed {removed} outlier customers (Monetary > £{q99:,.0f})')
print(f'✅ Clean RFM dataset: {len(rfm_clean):,} customers')

## ⚙️ Step 5 — Data Preprocessing (Scaling)

In [ ]:
# ⚖️ Standardize RFM features so K-Means isn't biased by scale
# (Monetary values are 100s-1000s; Recency/Frequency are much smaller)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_clean[['Recency', 'Frequency', 'Monetary']])

print('✅ Features standardized (mean=0, std=1)')
print(f'📏 Scaled array shape: {rfm_scaled.shape}')

# Quick sanity check — means should be ~0
means = rfm_scaled.mean(axis=0).round(4)
stds  = rfm_scaled.std(axis=0).round(4)
print(f'\n   Mean  → Recency: {means[0]}, Frequency: {means[1]}, Monetary: {means[2]}')
print(f'   Std   → Recency: {stds[0]},  Frequency: {stds[1]},  Monetary: {stds[2]}')

## 📈 Step 6 — Elbow Method + Silhouette Score
> Finding the optimal number of clusters (K)

In [ ]:
# 🔄 Compute WCSS (inertia) and Silhouette Score for K = 2 to 10
K_range = range(2, 11)
wcss = []
silhouette_scores = []

print('⏳ Running K-Means for K = 2 to 10...')
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    wcss.append(km.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))
    print(f'   K={k} → WCSS: {km.inertia_:,.0f} | Silhouette: {silhouette_scores[-1]:.4f}')

print('\n✅ Done!')

In [ ]:
# 📈 Elbow + Silhouette — side-by-side plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🔍 Finding Optimal K — Elbow Method & Silhouette Score', fontsize=15, fontweight='bold', color='white')

K_list = list(K_range)

# --- Left: Elbow curve ---
ax1.plot(K_list, wcss, 'o-', color='#38bdf8', linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2)
ax1.axvline(x=3, color='#f472b6', linestyle='--', linewidth=2, label='K = 3 (chosen)')
ax1.fill_between(K_list, wcss, alpha=0.1, color='#38bdf8')
ax1.set_title('📉 Elbow Method (WCSS)', fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Within-Cluster Sum of Squares')
ax1.set_xticks(K_list)
ax1.legend()

# --- Right: Silhouette score ---
bar_colors = ['#f472b6' if k == 3 else '#4ade80' for k in K_list]
bars = ax2.bar(K_list, silhouette_scores, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
ax2.set_title('📊 Silhouette Score per K', fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score (higher = better)')
ax2.set_xticks(K_list)

# Annotate bars
for bar, score in zip(bars, silhouette_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{score:.3f}', ha='center', va='bottom', fontsize=8, color='white')

plt.tight_layout()
plt.show()

best_k = K_list[silhouette_scores.index(max(silhouette_scores))]
print(f'\n📌 Best K by Silhouette Score: K = {best_k}')
print(f'📌 Chosen K for this analysis: K = 3 (business-interpretable elbow point)')

## 🤖 Step 7 — K-Means Clustering (K = 3)

In [ ]:
# 🤖 Fit final K-Means model with K = 3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm_clean['Cluster'] = kmeans.fit_predict(rfm_scaled)

# 🏷️ Map cluster numbers to meaningful business labels
cluster_means = rfm_clean.groupby('Cluster')['Monetary'].mean()
sorted_clusters = cluster_means.sort_values(ascending=False).index.tolist()

label_map = {
    sorted_clusters[0]: '🏆 Champions',
    sorted_clusters[1]: '🌱 Potential Loyalists',
    sorted_clusters[2]: '❄️  At-Risk / Dormant'
}
rfm_clean['Segment'] = rfm_clean['Cluster'].map(label_map)

# 📊 Distribution of customers per segment
dist = rfm_clean['Segment'].value_counts()
print('\n👥 Customers per Segment:')
for seg, count in dist.items():
    pct = count / len(rfm_clean) * 100
    print(f'   {seg}: {count:,} customers ({pct:.1f}%)')

## 🎨 Step 8 — Visualizations

In [ ]:
# 🎨 2D Scatter: Recency vs Monetary (colored by segment)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('🎯 Customer Segments — RFM Scatter Plots', fontsize=15, fontweight='bold', color='white')

colors = {'🏆 Champions': '#facc15', '🌱 Potential Loyalists': '#4ade80', '❄️  At-Risk / Dormant': '#60a5fa'}

for seg, color in colors.items():
    subset = rfm_clean[rfm_clean['Segment'] == seg]
    axes[0].scatter(subset['Recency'], subset['Monetary'], c=color, label=seg, alpha=0.5, s=20, edgecolors='none')
    axes[1].scatter(subset['Frequency'], subset['Monetary'], c=color, label=seg, alpha=0.5, s=20, edgecolors='none')

axes[0].set_title('⏱️ Recency  vs  💰 Monetary', fontweight='bold')
axes[0].set_xlabel('Recency (days since last purchase)')
axes[0].set_ylabel('Monetary (£ total spend)')
axes[0].legend()

axes[1].set_title('🔁 Frequency  vs  💰 Monetary', fontweight='bold')
axes[1].set_xlabel('Frequency (number of transactions)')
axes[1].set_ylabel('Monetary (£ total spend)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 📦 Box plots — RFM distribution per segment
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('📦 RFM Distribution by Customer Segment', fontsize=15, fontweight='bold', color='white')

metrics = [('Recency', '⏱️ Recency (days)', '#f472b6'),
           ('Frequency', '🔁 Frequency (transactions)', '#38bdf8'),
           ('Monetary', '💰 Monetary (£)', '#4ade80')]

segment_order = ['🏆 Champions', '🌱 Potential Loyalists', '❄️  At-Risk / Dormant']

for ax, (col, title, color) in zip(axes, metrics):
    data_by_seg = [rfm_clean[rfm_clean['Segment'] == seg][col].values for seg in segment_order]
    bp = ax.boxplot(data_by_seg, patch_artist=True, notch=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(['Champions', 'Potential\nLoyalists', 'At-Risk\n/ Dormant'], fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# 🕸️ Radar / Spider chart — Normalized RFM profile per segment
import matplotlib.patches as mpatches
from matplotlib.path import Path
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D

# Compute segment means and normalize to 0-1
seg_means = rfm_clean.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean()

# Invert Recency so that lower recency (more recent) = higher score on radar
seg_means_norm = seg_means.copy()
seg_means_norm['Recency'] = seg_means_norm['Recency'].max() - seg_means_norm['Recency']
seg_means_norm = (seg_means_norm - seg_means_norm.min()) / (seg_means_norm.max() - seg_means_norm.min())

categories = ['Recency\n(Inverted)', 'Frequency', 'Monetary']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#2a2a3e')
ax.set_title('🕸️ Segment RFM Profile (Normalized)', fontsize=14, fontweight='bold', color='white', pad=20)

segment_colors = {
    '🏆 Champions': '#facc15',
    '🌱 Potential Loyalists': '#4ade80',
    '❄️  At-Risk / Dormant': '#60a5fa'
}

for seg, color in segment_colors.items():
    if seg not in seg_means_norm.index:
        continue
    values = seg_means_norm.loc[seg].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=seg)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, color='white', size=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], color='gray', size=8)
ax.tick_params(colors='white')
ax.spines['polar'].set_color('#555577')
ax.grid(color='#444466', linestyle='--', alpha=0.6)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), labelcolor='white',
          facecolor='#1e1e2e', edgecolor='#555577')

plt.tight_layout()
plt.show()

In [ ]:
# 📊 Grouped bar chart — Segment summary (mean RFM per segment)
summary = rfm_clean.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
print('📋 Cluster Summary (Mean Values):')
print(summary.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('📊 Average RFM Values per Segment', fontsize=15, fontweight='bold', color='white')

metrics_info = [
    ('Recency',   '⏱️ Avg Recency (days)',         '#f472b6', 'Lower = Better'),
    ('Frequency', '🔁 Avg Frequency (transactions)', '#38bdf8', 'Higher = Better'),
    ('Monetary',  '💰 Avg Monetary (£)',             '#4ade80', 'Higher = Better'),
]

segment_order = ['🏆 Champions', '🌱 Potential Loyalists', '❄️  At-Risk / Dormant']

for ax, (col, title, color, note) in zip(axes, metrics_info):
    vals = [summary.loc[s, col] if s in summary.index else 0 for s in segment_order]
    bars = ax.bar(['Champions', 'Potential\nLoyalists', 'At-Risk\n/ Dormant'],
                  vals, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{title}\n({note})', fontweight='bold')
    ax.set_ylabel(col)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:,.1f}', ha='center', va='bottom', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 🍩 Donut charts — Customer count vs Revenue share per segment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('🍩 Segment Share — Customers vs Revenue', fontsize=15, fontweight='bold', color='white')
fig.patch.set_facecolor('#1e1e2e')

colors_pie = ['#facc15', '#4ade80', '#60a5fa']

# Customer count share
count_data = rfm_clean['Segment'].value_counts()
count_data = count_data.reindex(segment_order)

wedges, texts, autotexts = ax1.pie(
    count_data.values,
    labels=['Champions', 'Potential Loyalists', 'At-Risk / Dormant'],
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='#1e1e2e', linewidth=2)
)
ax1.set_facecolor('#1e1e2e')
ax1.set_title('👥 Customer Count Share', color='white', fontweight='bold')
for at in autotexts: at.set_color('white'); at.set_fontsize(10)
for t in texts: t.set_color('white'); t.set_fontsize(10)

# Revenue share
revenue_data = rfm_clean.groupby('Segment')['Monetary'].sum().reindex(segment_order)

wedges2, texts2, autotexts2 = ax2.pie(
    revenue_data.values,
    labels=['Champions', 'Potential Loyalists', 'At-Risk / Dormant'],
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='#1e1e2e', linewidth=2)
)
ax2.set_facecolor('#1e1e2e')
ax2.set_title('💰 Revenue Share', color='white', fontweight='bold')
for at in autotexts2: at.set_color('white'); at.set_fontsize(10)
for t in texts2: t.set_color('white'); t.set_fontsize(10)

plt.tight_layout()
plt.show()

In [ ]:
# 🌡️ Correlation heatmap on RFM features
fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle('🌡️ RFM Feature Correlation', fontsize=14, fontweight='bold', color='white')

corr = rfm_clean[['Recency', 'Frequency', 'Monetary']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=1,
    linecolor='#1e1e2e',
    annot_kws={'size': 12, 'weight': 'bold', 'color': 'black'}
)
ax.set_title('')

plt.tight_layout()
plt.show()

## 💡 Step 9 — Business Insights & Strategy

---

### 🏆 Champions
- **Recency:** ~23 days | **Frequency:** ~38 transactions | **Monetary:** ~£703
- Highly engaged, highest revenue contributors.
- ✅ **Strategy:** Loyalty rewards, VIP programs, early product access, brand ambassador opportunities.

---

### 🌱 Potential Loyalists
- **Recency:** ~47 days | **Frequency:** ~7.6 transactions | **Monetary:** ~£113
- Moderately active, have room to grow.
- ✅ **Strategy:** Personalized upsell offers, bundle deals, engagement campaigns, loyalty tier progression.

---

### ❄️ At-Risk / Dormant
- **Recency:** ~248 days | **Frequency:** ~3.2 transactions | **Monetary:** ~£54
- Largely inactive, at high churn risk.
- ✅ **Strategy:** Win-back email campaigns, discount offers, churn surveys, re-engagement nudges.

---

In [ ]:
# 📋 Final summary table
summary_final = rfm_clean.groupby('Segment').agg(
    Customer_Count=('Cluster', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).round(1)

summary_final['Revenue_Share_%'] = (summary_final['Total_Revenue'] / summary_final['Total_Revenue'].sum() * 100).round(1)

print('\n' + '=' * 80)
print('🎯  FINAL CUSTOMER SEGMENTATION SUMMARY')
print('=' * 80)
print(summary_final.to_string())
print('\n✅ Analysis complete!')